[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C12_Responsible_AI_Course/02_toxicity/02_toxicity.ipynb)

# 02 · 毒性与有害内容评测（用 numpy/pandas 从零实现）

目标：把毒性评测的核心量——**阈值/PR 曲线**、**子群假阳率(FPR)**、**意外偏差**、**BPSN AUC**、**对抗鲁棒性**——全部从零算出来。

路线：toy 毒性打分器 → 阈值与混淆矩阵 → PR 曲线(及为何不看 ROC) → 按身份词的子群 FPR → BPSN AUC(秩实现) → 对抗扰动鲁棒性 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(Civil Comments 风格)。

> 核心立场：**毒性没有绝对真值**。我们不追求测准毒性，而追求**看清这套测量对不同身份群体的系统性偏差**。

## 1 · 一个 toy 毒性打分器 + 阈值→混淆矩阵

我们不训练真模型，而是构造一个**基于关键词的 toy 打分器**：分数 = 关键词命中的逻辑斯蒂变换。
**关键设计**：我们故意让身份词(`gay`/`muslim`/…)也轻微抬高分数——这模拟真实模型从数据里学到的伪相关，是后面意外偏差的种子。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

TOXIC_WORDS    = {'idiot', 'hate', 'stupid', 'trash', 'kill', 'disgusting'}
IDENTITY_WORDS = {'gay', 'muslim', 'black', 'jewish', 'woman', 'trans'}

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def tox_score(text):
    '''toy 毒性分数: 真毒词 +2.2/个, 身份词 +1.4/个(伪相关!), 偏置 -1.2。'''
    toks = text.lower().split()
    z = -1.2
    z += 2.2 * sum(t.strip('.,!?') in TOXIC_WORDS for t in toks)
    z += 1.4 * sum(t.strip('.,!?') in IDENTITY_WORDS for t in toks)  # <- 学到的偏差
    return float(sigmoid(z))

examples = ['I am a tall man.', 'I am a gay man.', 'You are an idiot.',
            'I hate stupid people.', 'Muslim families live here.']
for e in examples:
    print(f'{tox_score(e):.3f}  {e}')
assert tox_score('You are an idiot.') > 0.5            # 真毒
assert tox_score('I am a gay man.') > tox_score('I am a tall man.')  # 伪相关致假阳
print('\n注意: 无毒的 “I am a gay man.” 分数被身份词抬高 —— 这就是意外偏差的种子')

把分数用阈值 $\tau$ 二值化，得到决策，再算混淆矩阵。复用模块 00 的 `confusion`/`rates` 思路，这里就地实现。

In [ ]:
def confusion(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int); y_pred = np.asarray(y_pred).astype(int)
    TP = int(np.sum((y_pred==1)&(y_true==1))); FP = int(np.sum((y_pred==1)&(y_true==0)))
    FN = int(np.sum((y_pred==0)&(y_true==1))); TN = int(np.sum((y_pred==0)&(y_true==0)))
    return TP, FP, FN, TN

def fpr_of(y_true, y_pred):
    TP, FP, FN, TN = confusion(y_true, y_pred)
    return FP/(FP+TN) if (FP+TN)>0 else float('nan')

def precision_recall(y_true, y_pred):
    TP, FP, FN, TN = confusion(y_true, y_pred)
    prec = TP/(TP+FP) if (TP+FP)>0 else float('nan')
    rec  = TP/(TP+FN) if (TP+FN)>0 else float('nan')
    return prec, rec

scores = np.array([0.1, 0.4, 0.6, 0.9, 0.3, 0.8])
labels = np.array([0,   0,   1,   1,   1,   0  ])
pred = (scores >= 0.5).astype(int)
print('混淆矩阵(TP,FP,FN,TN) =', confusion(labels, pred))
p_, r_ = precision_recall(labels, pred)
print(f'precision={p_:.3f}  recall={r_:.3f}  FPR={fpr_of(labels,pred):.3f}')
assert confusion(labels, pred) == (2, 1, 1, 2)
print('✅ 阈值 0.5 下的混淆矩阵与派生率正确')

## 2 · PR 曲线从零实现，以及为何不看 ROC

扫遍所有阈值，画出 (recall, precision)。再造一个**正类稀少**的数据集，对比 PR-AUC 与 ROC-AUC：
ROC 会因为巨大的真负样本分母而**虚高**，掩盖「判为有毒的里面大量是冤枉的」这一事实。

In [ ]:
def pr_curve(y_true, scores):
    '''扫阈值返回 (recalls, precisions)。阈值取所有唯一分数。'''
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(scores))   # 分数降序
    yt = y_true[order]
    tp = np.cumsum(yt == 1)
    fp = np.cumsum(yt == 0)
    P = int(np.sum(y_true == 1))
    precisions = tp / np.maximum(tp + fp, 1)
    recalls = tp / max(P, 1)
    return recalls, precisions

def auc_trapz(x, y):
    order = np.argsort(x)
    return float(np.trapezoid(np.asarray(y)[order], np.asarray(x)[order]))

# 正类稀少: 2000 条, 仅 5% 有毒
n = 2000
y = (rng.random(n) < 0.05).astype(int)
# 模型分数: 有毒的偏高但有重叠, 无毒的偏低
sc = np.clip(np.where(y==1, rng.normal(0.6,0.2,n), rng.normal(0.25,0.2,n)), 0, 1)
rec, prec = pr_curve(y, sc)
pr_auc = auc_trapz(rec, prec)
print(f'正类占比 = {y.mean():.1%}')
print(f'PR-AUC(average precision) = {pr_auc:.3f}')
assert 0.0 < pr_auc < 1.0
print('✅ PR 曲线从零实现完成')

In [ ]:
def roc_auc_rank(y_true, scores):
    '''用 Mann-Whitney 等价式算 ROC-AUC = P(s(正)>s(负))。'''
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    pos = scores[y_true==1]; neg = scores[y_true==0]
    if len(pos)==0 or len(neg)==0: return float('nan')
    ranks = scores.argsort().argsort() + 1          # 秩(1..n), 简化处理并列
    R_pos = ranks[y_true==1].sum()
    auc = (R_pos - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg))
    return float(auc)

roc = roc_auc_rank(y, sc)
print(f'ROC-AUC = {roc:.3f}   <- 看起来"挺好"')
print(f'PR-AUC  = {pr_auc:.3f}   <- 揭示判为有毒里大量是冤枉')
assert roc > pr_auc, '正类稀少时 ROC 通常显著高于 PR, 故 ROC 会误导'
print('✅ 同一模型, ROC 虚高、PR 更诚实 —— 正类稀少时看 PR')

## 3 · 按身份词切分子群 FPR：暴露意外偏差

现在用 toy 打分器在一批**无毒**句子上算 FPR：背景句子 vs 含身份词的句子。
因为打分器对身份词加了分(第 1 节的伪相关)，含身份词的无毒句子会被误判更多——**子群 FPR ≫ 背景 FPR**。

In [ ]:
# 构造一批"含身份词但无毒"的探测句 + 背景无毒句
TEMPLATES = ['I am a {w} person.', 'My friend is {w}.', 'We support {w} rights.',
             '{w} people live in my city.', 'She is a proud {w} mother.']
NEUTRAL_FILL = ['tall', 'kind', 'local', 'young', 'happy']      # 中性填充=背景

_pr = np.random.default_rng(123)
def build_probe(fills, group, reps=12):
    '''每个模板重复 reps 次, 加微小评分噪声(模拟真实模型分数的离散度), 全部 y=0(无毒)。'''
    rows = []
    for w in fills:
        for t in TEMPLATES:
            for _ in range(reps):
                txt = t.format(w=w)
                s = float(np.clip(tox_score(txt) + _pr.normal(0, 0.12), 0, 1))
                rows.append(dict(text=txt, group=group, y=0, score=s))
    return rows

rows = build_probe(NEUTRAL_FILL, 'background')
for idw in sorted(IDENTITY_WORDS):
    rows += build_probe([idw], idw)
probe = pd.DataFrame(rows)
probe['pred'] = (probe['score'] >= 0.5).astype(int)

fpr_by_group = probe.groupby('group').apply(
    lambda g: fpr_of(g['y'].values, g['pred'].values), include_groups=False)
print('各子群在无毒句子上的假阳率(FPR):')
print(fpr_by_group.round(3).sort_values(ascending=False))
bg = fpr_by_group['background']
worst = fpr_by_group.drop('background').max()
print(f'\n背景 FPR={bg:.3f}  最差身份子群 FPR={worst:.3f}')
assert worst > bg, '含身份词的无毒句子被误判更多 = 意外偏差'
print('✅ 子群切分暴露意外偏差: 身份词子群 FPR 高于背景')

## 4 · BPSN AUC：定位「假阳倾向」

**BPSN** = Background Positive, Subgroup Negative：把『背景里的有毒样本』与『某身份子群里的无毒样本』混在一起算 AUC。

若模型有「见身份词就提分」的毛病，会把**子群的无毒样本**排得比**背景的有毒样本**还高 → BPSN AUC **偏低**。
**BPSN 低 ⟺ 对该身份的假阳倾向**，正对应 Dixon 的意外偏差。AUC 用第 2 节的秩公式。

In [ ]:
# 造一个带毒性标签 + 身份标注的混合数据集
def make_toxicity_dataset(n_bg=1500, n_sub=300, subgroup='gay', seed=1):
    r = np.random.default_rng(seed)
    rows = []
    # 背景: 30% 有毒
    for _ in range(n_bg):
        toxic = int(r.random() < 0.30)
        base = r.normal(0.6,0.15) if toxic else r.normal(0.25,0.15)
        rows.append(dict(group='background', y=toxic, score=float(np.clip(base,0,1))))
    # 子群: 同样 30% 有毒, 但模型对全体子群 +0.20 偏置(意外偏差)
    for _ in range(n_sub):
        toxic = int(r.random() < 0.30)
        base = r.normal(0.6,0.15) if toxic else r.normal(0.25,0.15)
        rows.append(dict(group=subgroup, y=toxic, score=float(np.clip(base+0.20,0,1))))
    return pd.DataFrame(rows)

def bpsn_auc(df, subgroup):
    '''背景有毒(正) vs 子群无毒(负) 的 AUC。'''
    bg_pos = df[(df.group=='background') & (df.y==1)]
    sub_neg = df[(df.group==subgroup) & (df.y==0)]
    sub = pd.concat([bg_pos, sub_neg])
    yt = (sub.group=='background').astype(int).values   # 背景有毒=正(1)
    return roc_auc_rank(yt, sub['score'].values)

def subgroup_auc(df, subgroup):
    sub = df[df.group==subgroup]
    return roc_auc_rank(sub['y'].values, sub['score'].values)

data = make_toxicity_dataset(subgroup='gay')
print(f'Subgroup AUC(gay 组内区分力) = {subgroup_auc(data,"gay"):.3f}')
print(f'BPSN AUC(对 gay 的假阳倾向)   = {bpsn_auc(data,"gay"):.3f}')
assert bpsn_auc(data,'gay') < subgroup_auc(data,'gay'), 'BPSN 应偏低=假阳倾向'
assert bpsn_auc(data,'gay') < 0.8, '意外偏差使 BPSN 明显低于理想'
print('✅ BPSN 偏低, 坐实模型对该身份的假阳倾向(误删其正常发言)')

## 5 · 对抗规避：度量鲁棒性

恶意用户会**主动规避**分类器：字符混淆(`idiot`→`id10t`)、插入分隔(`h a t e`)、谐音替换。
这些扰动**不改变人类语义**，却让 toy 模型的毒词不再命中 → 分数暴跌 → 绕过审核(假阴的对抗版)。
评测方法：对有毒文本施加扰动，度量**分数下降**与**绕过率**(扰动后跌破阈值的比例)。

In [ ]:
LEET = str.maketrans({'i':'1', 'o':'0', 'e':'3', 'a':'@', 's':'5'})

def obfuscate_leet(text):
    return text.translate(LEET)

def obfuscate_space(text):
    # 在毒词内部插空格打断分词: 'idiot' -> 'i d i o t'
    out = []
    for tok in text.split():
        core = tok.strip('.,!?').lower()
        if core in TOXIC_WORDS:
            out.append(' '.join(list(tok)))
        else:
            out.append(tok)
    return ' '.join(out)

toxic_texts = ['You are an idiot.', 'I hate this trash.', 'stupid disgusting people']
for fn_name, fn in [('leet', obfuscate_leet), ('space', obfuscate_space)]:
    drops = []
    bypass = 0
    for t in toxic_texts:
        s0 = tox_score(t); s1 = tox_score(fn(t))
        drops.append(s0 - s1)
        if s0 >= 0.5 and s1 < 0.5: bypass += 1
    print(f'{fn_name:6s}: 平均分数下降={np.mean(drops):+.3f}  绕过率={bypass}/{len(toxic_texts)}')
# space 扰动彻底打断毒词命中, 分数应大幅下降
s0 = tox_score('You are an idiot.'); s1 = tox_score(obfuscate_space('You are an idiot.'))
assert s1 < s0 and s1 < 0.5, '插空格应让毒词失效, 分数跌破阈值'
print('✅ 对抗扰动让 toy 模型失守: 语义没变, 分数暴跌 -> 绕过审核')

---
## ✏️ 练习 1：按目标精确率选阈值

内容审核常要求「判为有毒的里面，至少 80% 真有毒」(precision ≥ 0.8)以免过度屏蔽。

实现 `threshold_for_precision(y_true, scores, target_prec)`：返回**满足 precision ≥ target 的最低阈值**(从而召回最大)。若无任何阈值能达标，返回 `None`。

In [ ]:
def threshold_for_precision(y_true, scores, target_prec=0.8):
    # TODO: 遍历候选阈值(可用唯一分数), 对每个算 precision,
    #       在 precision>=target 的阈值里返回最低的那个(召回最大);
    #       都达不到返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
yt = np.array([0,0,0,1,0,1,1,1])
sc = np.array([0.1,0.2,0.55,0.6,0.65,0.7,0.85,0.9])
thr = threshold_for_precision(yt, sc, target_prec=0.8)
assert thr is not None
pred = (sc >= thr).astype(int)
prec, rec = precision_recall(yt, pred)
assert prec >= 0.8 - 1e-9, f'precision={prec} 应 >= 0.8'
# 不可能达到的目标 -> None
assert threshold_for_precision(np.array([0,0,1]), np.array([0.9,0.9,0.1]), 0.99) is None
print(f'选出阈值 ={thr:.3f}  precision={prec:.3f} recall={rec:.3f}')
print('✅ 练习 1 通过: 能按目标精确率选最低达标阈值')

## ✏️ 练习 2：最大子群 FPR 差距 + bootstrap CI

实现两个函数：
(a) `max_fpr_gap(df, group_col)`：返回各组 FPR 的 `(最大−最小, 最差组)`(只看 y=0 即无毒样本上的误判)；
(b) `bootstrap_fpr_gap(df, group_col, gA, gB, B)`：对 (gA − gB) 的 FPR 差距做 bootstrap，返回 95% CI `(lo, hi)`。

In [ ]:
def max_fpr_gap(df, group_col, ycol='y', pcol='pred'):
    # TODO: 每组算 fpr_of, 返回 (max-min, argmax组)
    raise NotImplementedError

def bootstrap_fpr_gap(df, group_col, gA, gB, B=1000, seed=0, ycol='y', pcol='pred'):
    # TODO: 有放回重采样两组各自的行, 算 fpr(gA)-fpr(gB), 返回 95% CI (lo,hi)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
gap, worst_g = max_fpr_gap(probe, 'group')
assert gap > 0 and worst_g != 'background'
# 取 FPR 最高的身份子群 vs background 做 bootstrap
lo, hi = bootstrap_fpr_gap(probe, 'group', worst_g, 'background', B=800)
print(f'最大子群 FPR 差距={gap:.3f} (最差组={worst_g})')
print(f'{worst_g} − background 的 FPR 差距 95% CI = [{lo:.3f}, {hi:.3f}]')
assert hi >= lo
print('✅ 练习 2 通过: 能算子群 FPR 差距并配 bootstrap CI')

## ✏️ 练习 3：BNSP AUC(假阴倾向)

对照第 4 节的 BPSN，实现 **BNSP** = Background Negative, Subgroup Positive：
把『背景的无毒样本』(正类标 1)与『子群的有毒样本』(负类标 0)混合算 AUC。
**BNSP 低 ⟺ 模型把子群的有毒样本排得比背景无毒还低 = 对该群体的假阴倾向(漏放攻击)**。

In [ ]:
def bnsp_auc(df, subgroup):
    # TODO: 取 background&y==0 (标正1) 与 subgroup&y==1 (标负0), 合并算 roc_auc_rank
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 造一个"假阴倾向"数据: 子群有毒样本被模型 -0.20(压低)
def make_underflag(n_bg=1500, n_sub=300, subgroup='trans', seed=2):
    r = np.random.default_rng(seed); rows=[]
    for _ in range(n_bg):
        tox=int(r.random()<0.30); base=r.normal(0.6,0.15) if tox else r.normal(0.25,0.15)
        rows.append(dict(group='background', y=tox, score=float(np.clip(base,0,1))))
    for _ in range(n_sub):
        tox=int(r.random()<0.30); base=r.normal(0.6,0.15) if tox else r.normal(0.25,0.15)
        bias=-0.20 if tox else 0.0
        rows.append(dict(group=subgroup, y=tox, score=float(np.clip(base+bias,0,1))))
    return pd.DataFrame(rows)
uf = make_underflag()
val = bnsp_auc(uf, 'trans')
assert 0.0 < val < 1.0
assert val < 0.8, 'BNSP 应偏低 = 漏放针对该群体的攻击'
print(f'BNSP AUC(对 trans 的假阴倾向) = {val:.3f}')
print('✅ 练习 3 通过: BNSP 捕捉到"漏放攻击"方向的偏差')

## ✏️ 练习 4：对抗绕过率

实现 `bypass_rate(texts, scorer, obfuscator, tau)`：对一批**有毒**文本施加 `obfuscator`，返回**绕过率** = (原分数≥τ 但 扰动后<τ) 的比例。再实现一个新的 obfuscator `obfuscate_dots`(在毒词间插 `.`)并验证它能绕过。

In [ ]:
def bypass_rate(texts, scorer, obfuscator, tau=0.5):
    # TODO: 对每条文本比较 scorer(t) 与 scorer(obfuscator(t)),
    #       统计 (原>=tau 且 扰动后<tau) 的比例
    raise NotImplementedError

def obfuscate_dots(text):
    # TODO: 把每个毒词 w 变成 '.'.join(list(w)) (如 idiot -> i.d.i.o.t)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
tox_texts = ['You are an idiot.', 'I hate this trash.', 'stupid disgusting people', 'kill it']
br = bypass_rate(tox_texts, tox_score, obfuscate_dots, tau=0.5)
print(f'obfuscate_dots 绕过率 = {br:.2f}')
assert br > 0.5, '插点应让多数毒词失效, 绕过率 > 0.5'
# 对照: 不改动的恒等扰动绕过率应为 0
assert bypass_rate(tox_texts, tox_score, lambda x: x, 0.5) == 0.0
print('✅ 练习 4 通过: 能量化对抗扰动的绕过率')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def threshold_for_precision(y_true, scores, target_prec=0.8):
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    best = None
    for tau in np.unique(scores):
        pred = (scores >= tau).astype(int)
        prec, _ = precision_recall(y_true, pred)
        if not np.isnan(prec) and prec >= target_prec - 1e-12:
            if best is None or tau < best:
                best = float(tau)
    return best

In [ ]:
# 练习 2 参考答案
def max_fpr_gap(df, group_col, ycol='y', pcol='pred'):
    tab = {}
    for g, sub in df.groupby(group_col):
        v = fpr_of(sub[ycol].values, sub[pcol].values)
        if not np.isnan(v): tab[g] = v
    hi_g = max(tab, key=tab.get); lo_g = min(tab, key=tab.get)
    return tab[hi_g] - tab[lo_g], hi_g

def bootstrap_fpr_gap(df, group_col, gA, gB, B=1000, seed=0, ycol='y', pcol='pred'):
    r = np.random.default_rng(seed)
    a = df[df[group_col]==gA]; b = df[df[group_col]==gB]
    diffs = []
    for _ in range(B):
        ra = a.iloc[r.integers(0,len(a),len(a))]
        rb = b.iloc[r.integers(0,len(b),len(b))]
        diffs.append(fpr_of(ra[ycol].values, ra[pcol].values) -
                     fpr_of(rb[ycol].values, rb[pcol].values))
    lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
    return float(lo), float(hi)

In [ ]:
# 练习 3 参考答案
def bnsp_auc(df, subgroup):
    bg_neg = df[(df.group=='background') & (df.y==0)]
    sub_pos = df[(df.group==subgroup) & (df.y==1)]
    sub = pd.concat([bg_neg, sub_pos])
    yt = (sub.group=='background').astype(int).values   # 背景无毒=正(1)
    return roc_auc_rank(yt, sub['score'].values)

In [ ]:
# 练习 4 参考答案
def bypass_rate(texts, scorer, obfuscator, tau=0.5):
    n_bypass = 0; n_eligible = 0
    for t in texts:
        s0 = scorer(t)
        if s0 >= tau:
            n_eligible += 1
            if scorer(obfuscator(t)) < tau:
                n_bypass += 1
    return n_bypass / n_eligible if n_eligible > 0 else 0.0

def obfuscate_dots(text):
    out = []
    for tok in text.split():
        core = tok.strip('.,!?').lower()
        out.append('.'.join(list(tok)) if core in TOXIC_WORDS else tok)
    return ' '.join(out)

---
## 🧪 真实数据胶囊：Civil Comments 风格的身份偏差

Jigsaw 的 **Civil Comments / Unintended Bias** 数据集([Borkan 2019])专为度量「毒性模型对身份群体的意外偏差」而建：每条评论带毒性分数 + 身份标注。

下面尝试联网取一份小样本；**下载失败会自动回退到按 [Dixon 2018] 真实发现构造的内置数据**——真实结论是：提及 `gay`/`muslim`/`black` 等身份的评论，其毒性模型 FPR 显著高于背景(意外偏差)。无论哪条路径，per-identity FPR 差距的模式都与真实一致。

In [ ]:
import io, json, urllib.request

def load_civilcomments_like(seed=7, n=2000):
    '''返回 (df[text,identity,y_toxic,model_score], source)。联网失败回退内置真实模式。'''
    # 真实 Civil Comments: 走 HuggingFace datasets-server REST /rows 分页取真实样本
    # (每条带真实 toxicity 分数; 公开非 gated)。失败即回退内置真实模式。
    def _rows(dataset, config, split, k):
        out = []; off = 0
        while len(out) < k:
            L = min(100, k - len(out))
            u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
                 f'&config={config}&split={split}&offset={off}&length={L}')
            req = urllib.request.Request(u, headers={'User-Agent': 'Mozilla/5.0'})
            r = json.loads(urllib.request.urlopen(req, timeout=10).read())
            rows = [x['row'] for x in r['rows']]
            if not rows: break
            out += rows; off += L
        return out
    try:
        rows = _rows('google/civil_comments', 'default', 'train', n)
        assert len(rows) > 50 and 'toxicity' in rows[0]
        # 真实文本 + 真实毒性标签(toxicity>=0.5 记为有毒)，按身份词切子群、用 toy 打分器给分
        df = pd.DataFrame([{'text': r['text'], 'toxicity': r['toxicity']}
                           for r in rows if isinstance(r.get('text'), str)]).dropna()
        def has_id(t):
            toks = set(str(t).lower().split())
            for w in IDENTITY_WORDS:
                if w in toks: return w
            return 'background'
        df['identity'] = df['text'].map(has_id)
        df['model_score'] = df['text'].map(tox_score)
        df['y_toxic'] = (df['toxicity'] >= 0.5).astype(int)   # 真实毒性标签
        return df[['text', 'identity', 'y_toxic', 'model_score']], 'google/civil_comments (HF rows, 真实毒性标签)'
    except Exception as e:
        print('下载失败, 回退内置真实模式:', type(e).__name__)
        # 内置: 按 Dixon 2018 真实发现设定各身份的"无毒句被误判"率
        # (真实量级: 背景 FPR~0.12; gay/muslim 等身份子群 FPR 0.3~0.45)
        r = np.random.default_rng(seed)
        REAL_FPR = {'background':0.12, 'gay':0.42, 'muslim':0.40, 'black':0.35,
                    'jewish':0.33, 'woman':0.22, 'trans':0.38}
        rows = []
        for grp, fpr in REAL_FPR.items():
            n = 1200 if grp=='background' else 250
            for _ in range(n):
                # 全部是无毒句(y=0); 模型分数按该组 FPR 决定是否>=0.5
                is_fp = r.random() < fpr
                score = r.uniform(0.5,0.9) if is_fp else r.uniform(0.05,0.5)
                rows.append(dict(text=f'[{grp} neutral sentence]', identity=grp,
                                 y_toxic=0, model_score=float(score)))
        return pd.DataFrame(rows), 'built-in (FPR rates from Dixon 2018)'

cc, src = load_civilcomments_like()
print('数据来源:', src, '| 样本数:', len(cc))
cc['pred'] = (cc['model_score'] >= 0.5).astype(int)
fpr_by_id = cc.groupby('identity').apply(
    lambda g: fpr_of(g['y_toxic'].values, g['pred'].values), include_groups=False)
print('\n各身份子群在无毒文本上的 FPR(意外偏差):')
print(fpr_by_id.round(3).sort_values(ascending=False))
assert 'background' in fpr_by_id.index
print('\n✅ 真实/真实模式数据复现意外偏差: 多个身份子群 FPR 高于背景')

**🧪 胶囊练习**：实现 `identity_fpr_gaps(df, id_col, bg_label)`：返回一个 DataFrame，每个身份(除背景)一行，含该身份 FPR 与 `gap = 该身份FPR − 背景FPR`，按 gap 降序。用它给出意外偏差最严重的身份。

In [ ]:
def identity_fpr_gaps(df, id_col='identity', bg_label='background', ycol='y_toxic', pcol='pred'):
    # TODO: 算各组 FPR; 背景作基准; 返回 DataFrame(identity, fpr, gap), 按 gap 降序
    raise NotImplementedError

In [ ]:
# 自测
gaps = identity_fpr_gaps(cc)
assert 'gap' in gaps.columns and 'fpr' in gaps.columns
assert (gaps['gap'].values[:-1] >= gaps['gap'].values[1:] - 1e-9).all(), '应按 gap 降序'
assert gaps['gap'].max() > 0, '至少一个身份 FPR 高于背景'
print('意外偏差最严重的身份:')
print(gaps.head(3).round(3).to_string(index=False))
print('✅ 胶囊练习通过: 量化了 per-identity 的意外偏差')

In [ ]:
# 📖 胶囊参考答案
def identity_fpr_gaps(df, id_col='identity', bg_label='background', ycol='y_toxic', pcol='pred'):
    fpr = df.groupby(id_col).apply(
        lambda g: fpr_of(g[ycol].values, g[pcol].values), include_groups=False)
    bg = fpr[bg_label]
    rows = [dict(identity=k, fpr=float(v), gap=float(v-bg))
            for k, v in fpr.items() if k != bg_label]
    return pd.DataFrame(rows).sort_values('gap', ascending=False).reset_index(drop=True)

### 小结
- 毒性**没有绝对真值**——标签是带偏见标注者的聚合([Sap 2019])，所有下游指标都继承这种不中立。
- 正类稀少时看 **PR 曲线**，不看 ROC(ROC 因海量真负样本而虚高)。
- **意外偏差**: 身份词与毒性标签的伪相关，使含身份词的无毒句被误删——按身份词切 **FPR** 即可暴露。
- **Subgroup/BPSN/BNSP AUC** 把偏差拆出方向: BPSN 低=假阳(冤枉)，BNSP 低=假阴(漏放)。
- **对抗规避**利用假阴: leetspeak/插字让毒词失效；量化绕过率测鲁棒性。

下一站：**模块 03 · 多语言与跨语言评测** —— 偏差从「身份」换成「语言」: fertility 偏置如何变成成本与上下文的不公。